The lab 2 used parallelization workflow parttern to solve the problem of comming up with a challenge and getting solution.

The flow used was

1. User input (come up with a challenge)
2. Response (the challenge is generated by the model)
3. then code was used to send the task to 3 different llms
4. code was used to combine the result
5. The result was sent out to another llm
6. Final output

Because code was used to output to a number of llms, this is parallelization


The goal of this submission is to use the orchestrator model

first the imports

In [ ]:

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from anthropic import Anthropic

loading environment variables

In [ ]:
load_dotenv(override=True)

Load API keys for the various models

In [ ]:
# Print the key prefixes to help with any debugging

openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

Setup a tool that calls the different services depending on the service name passed

In [ ]:
llm_services = ["openai", "anthropic", "google", "llama3"]

# OpenAI Chat Completions / Responses `tools=[...]` entry for the orchestrator model
llm_service_tool_definition = {
    "type": "function",
    "function": {
        "name": "llm_service_tool",
        "description": (
            "Send a prompt to a worker LLM. Use openai, google (Gemini), or llama3 (Groq) "
            "via the OpenAI-compatible client; anthropic uses the Anthropic API. "
            "Always returns JSON with keys service (canonical name used) and response (model text); "
            "on failure service and response are null and error explains why."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "service_name": {
                    "type": "string",
                    "enum": llm_services,
                    "description": "Which provider to call.",
                },
                "user_message": {
                    "type": "string",
                    "description": "The user prompt or task for that model.",
                },
            },
            "required": ["service_name", "user_message"],
        },
    },
}


def _llm_tool_result(
    *,
    service: str | None,
    response: str | None = None,
    error: str | None = None,
) -> str:
    payload = {"service": service, "response": response}
    if error is not None:
        payload["error"] = error
    return json.dumps(payload)


def llm_service_tool(service_name: str, user_message: str) -> str:
    print("tool call with service name", service_name)
    print("tool call with user message:", user_message)
    """Worker backend for orchestrator tool calls: Anthropic SDK for anthropic, OpenAI client for all others."""
    name = (service_name or "").strip().lower()
    if name not in llm_services:
        return _llm_tool_result(
            service=None,
            response=None,
            error=f"Unknown service_name {service_name!r}. Choose one of: {llm_services}",
        )

    messages = [{"role": "user", "content": user_message}]

    if name == "anthropic":
        if not anthropic_api_key:
            return _llm_tool_result(
                service=None, response=None, error="ANTHROPIC_API_KEY is not set."
            )
        client = Anthropic()
        api_response = client.messages.create(
            model="claude-sonnet-4-5",
            max_tokens=8192,
            messages=messages,
        )
        return _llm_tool_result(service=name, response=api_response.content[0].text)

    if name == "openai":
        if not openai_api_key:
            return _llm_tool_result(
                service=None, response=None, error="OPENAI_API_KEY is not set."
            )
        client = OpenAI()
        completion = client.chat.completions.create(
            model="gpt-5-nano",
            messages=messages,
        )
    elif name == "google":
        if not google_api_key:
            return _llm_tool_result(
                service=None, response=None, error="GOOGLE_API_KEY is not set."
            )
        client = OpenAI(
            api_key=google_api_key,
            base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
        )
        completion = client.chat.completions.create(
            model="gemini-2.5-flash",
            messages=messages,
        )
    elif name == "llama3":
        if not groq_api_key:
            return _llm_tool_result(
                service=None, response=None, error="GROQ_API_KEY is not set."
            )
        client = OpenAI(
            api_key=groq_api_key,
            base_url="https://api.groq.com/openai/v1",
        )
        completion = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=messages,
        )
    else:
        return _llm_tool_result(
            service=None, response=None, error=f"Unhandled service {name!r}"
        )

    text = completion.choices[0].message.content or ""
    return _llm_tool_result(service=name, response=text)


Test the tool

In [ ]:
llm_service_tool('Anthropic','You are an advisor to a small island nation of 3 million people whose economy depends 30% on tourism and 20% on agriculture; it faces projected sea-level rise and more intense storms, has high income inequality, and a total public-budget capacity of $20 billion to spend over the next 10 years. Design a concrete, prioritized 10-year climate-resilience and economic-transition plan that meets these goals simultaneously: (1) protect at least 80% of the population from coastal flooding under a 0.5‑meter sea‑level rise and 100‑year storm surge within 10 years; (2) reduce national greenhouse‑gas emissions by 40% from current levels within 10 years; and (3) avoid net job losses and reduce income inequality (give a plausible inequality metric improvement). For your answer, provide: (A) the top 6 interventions (policy, infrastructure, market, and social programs) in order of priority and why; (B) a year-by-year implementation timeline and a per-intervention budget allocation summing to ≤ $20B, with transparent assumptions; (C) rough quantified estimates (with assumptions) of emissions reductions and net jobs created/lost per intervention over 10 years and the expected change in an inequality metric (e.g., Gini or Palma); (D) three measurable KPIs to track progress and baselines for each; (E) the most important political, ethical, and distributional trade-offs and how you would mitigate them; (F) three realistic failure modes (technical, political, financial) and contingency plans for each; and (G) the three specific additional data points or analyses that would most likely change your plan and why. Be quantitative where possible; avoid vague platitudes.')
llm_service_tool('openai','You are an advisor to a small island nation of 3 million people whose economy depends 30% on tourism and 20% on agriculture; it faces projected sea-level rise and more intense storms, has high income inequality, and a total public-budget capacity of $20 billion to spend over the next 10 years. Design a concrete, prioritized 10-year climate-resilience and economic-transition plan that meets these goals simultaneously: (1) protect at least 80% of the population from coastal flooding under a 0.5‑meter sea‑level rise and 100‑year storm surge within 10 years; (2) reduce national greenhouse‑gas emissions by 40% from current levels within 10 years; and (3) avoid net job losses and reduce income inequality (give a plausible inequality metric improvement). For your answer, provide: (A) the top 6 interventions (policy, infrastructure, market, and social programs) in order of priority and why; (B) a year-by-year implementation timeline and a per-intervention budget allocation summing to ≤ $20B, with transparent assumptions; (C) rough quantified estimates (with assumptions) of emissions reductions and net jobs created/lost per intervention over 10 years and the expected change in an inequality metric (e.g., Gini or Palma); (D) three measurable KPIs to track progress and baselines for each; (E) the most important political, ethical, and distributional trade-offs and how you would mitigate them; (F) three realistic failure modes (technical, political, financial) and contingency plans for each; and (G) the three specific additional data points or analyses that would most likely change your plan and why. Be quantitative where possible; avoid vague platitudes.')
llm_service_tool('gemini','You are an advisor to a small island nation of 3 million people whose economy depends 30% on tourism and 20% on agriculture; it faces projected sea-level rise and more intense storms, has high income inequality, and a total public-budget capacity of $20 billion to spend over the next 10 years. Design a concrete, prioritized 10-year climate-resilience and economic-transition plan that meets these goals simultaneously: (1) protect at least 80% of the population from coastal flooding under a 0.5‑meter sea‑level rise and 100‑year storm surge within 10 years; (2) reduce national greenhouse‑gas emissions by 40% from current levels within 10 years; and (3) avoid net job losses and reduce income inequality (give a plausible inequality metric improvement). For your answer, provide: (A) the top 6 interventions (policy, infrastructure, market, and social programs) in order of priority and why; (B) a year-by-year implementation timeline and a per-intervention budget allocation summing to ≤ $20B, with transparent assumptions; (C) rough quantified estimates (with assumptions) of emissions reductions and net jobs created/lost per intervention over 10 years and the expected change in an inequality metric (e.g., Gini or Palma); (D) three measurable KPIs to track progress and baselines for each; (E) the most important political, ethical, and distributional trade-offs and how you would mitigate them; (F) three realistic failure modes (technical, political, financial) and contingency plans for each; and (G) the three specific additional data points or analyses that would most likely change your plan and why. Be quantitative where possible; avoid vague platitudes.')
llm_service_tool('llama3','You are an advisor to a small island nation of 3 million people whose economy depends 30% on tourism and 20% on agriculture; it faces projected sea-level rise and more intense storms, has high income inequality, and a total public-budget capacity of $20 billion to spend over the next 10 years. Design a concrete, prioritized 10-year climate-resilience and economic-transition plan that meets these goals simultaneously: (1) protect at least 80% of the population from coastal flooding under a 0.5‑meter sea‑level rise and 100‑year storm surge within 10 years; (2) reduce national greenhouse‑gas emissions by 40% from current levels within 10 years; and (3) avoid net job losses and reduce income inequality (give a plausible inequality metric improvement). For your answer, provide: (A) the top 6 interventions (policy, infrastructure, market, and social programs) in order of priority and why; (B) a year-by-year implementation timeline and a per-intervention budget allocation summing to ≤ $20B, with transparent assumptions; (C) rough quantified estimates (with assumptions) of emissions reductions and net jobs created/lost per intervention over 10 years and the expected change in an inequality metric (e.g., Gini or Palma); (D) three measurable KPIs to track progress and baselines for each; (E) the most important political, ethical, and distributional trade-offs and how you would mitigate them; (F) three realistic failure modes (technical, political, financial) and contingency plans for each; and (G) the three specific additional data points or analyses that would most likely change your plan and why. Be quantitative where possible; avoid vague platitudes.')

lets define the function that gets called to trigger the process

In [ ]:
def trigger_llm_service(user_message):
    request = f"""
    You are an expert in orchestration of LLMs.
    You have access to the following LLMs:
    {llm_services} through tools provided to you.

    You have been given a question: {user_message}

    First, come up with an answer to the question. Your answer should be an instruction that is passed to the LLMs for further processing.
    
    Now make use of the llms provided to you to get the responses.

    Finally, when you have responses from the LLMs, proceed to come up with a final response which ranks the LLMs based on the responses, the final response should be in JSON format only, no explanation.
    """
    messages = [{"role": "user", "content": request}]
    openai = OpenAI()
    response = openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=[llm_service_tool_definition]
    )
    while response.choices[0].finish_reason == "tool_calls":
        print("tool call size:", len(response.choices[0].message.tool_calls))
        messages.append(response.choices[0].message)
        tool_call_responses = []
        for tool_call in response.choices[0].message.tool_calls:
            arguments = json.loads(tool_call.function.arguments)
            service = arguments["service_name"]
            message = arguments["user_message"]
            response = llm_service_tool(service, message)
            tool_call_responses.append({"content":response, "tool_call_id": tool_call.id, "tool_call_name": tool_call.function.name,"role":"tool"})
        messages.extend(tool_call_responses)
        response = openai.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=[llm_service_tool_definition]
        )
    print("Final response", response.choices[0].message.content)
    response.choices[0].message.content

lets put it all to test

In [ ]:
response = trigger_llm_service("Please come up with a challenging, nuanced question that I can ask a number of LLMs to evaluate their intelligence.")
response